# hb1603_survey_echogram_pyramid

The whole HB1603 cruise as an EchogramGL pyramid: per-file MVBS on depth, 18 and 38 kHz, seabed masked by the phase-aware picker.

**Author:** <Your Name>
**Version:** 1.0

In [ ]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_calibration.git
# %pip install echopype>=0.6.0
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_ML.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_EchogramGL.git

In [ ]:
from aa_si_utils.data_retrieval import query_ncei_data
from aa_si_calibration.calibration import read_raw_file_config
from aa_si_calibration.calibration import record_raw_file_configs
from aa_si_calibration.calibration import standardize_calibration_files
from aa_si_calibration.calibration import build_calibration_mapping
from aa_si_utils.utils import read_raw_files_to_stores
from aa_si_utils.utils import combine_raw_stores
from aa_si_calibration.calibration import extract_standardized_calibration_parameters
from echopype.calibrate import compute_Sv
from aa_si_utils.utils import compute_transducer_depth
from echopype.consolidate import add_depth
from aa_si_utils.utils import crop_range_samples
from echopype.consolidate import add_splitbeam_angle
from aa_si_utils.seabed import detect_seafloor_phase
from aa_si_utils.utils import create_seafloor_mask
from aa_si_utils.utils import create_surface_mask
from aa_si_utils.utils import create_frequency_mask
from aa_si_utils.utils import combine_masks
from aa_si_utils.utils import apply_mask_to_sv
from aa_si_ml.ml import remove_noise
from aa_si_utils.utils import mask_sparse_bins
from echopype.commongrid import compute_MVBS
from aa_si_utils.utils import concat_datasets
from aa_si_utils.utils import rechunk_dataset
from aa_si_echogram_gl.ops import plan_pyramid_segments
from aa_si_echogram_gl.ops import build_echogram_pyramid
from aa_si_echogram_gl.ops import merge_echogram_pyramids


In [ ]:
file_time_start = '2016-06-01T00:00:00'  # str
file_time_end = '2016-09-01T00:00:00'  # str
cal_input_folder = '../calibration_files/HB201607_cal'  # path
calibration_output_dir = './outputs/calibration'  # path
crop_max_range_m = 1800.0  # float
viewer_crop_max_range_m = 3000.0  # float
viewer_grid_max_depth = '3010m'  # str
cruise_name = 'HB1603'  # str
cruise_id = 'HB1603'  # str
record_author = '<Your Name>'  # str
max_scan_bytes = 1048576  # int
match_tolerances = {'transmit_power': 1001.0}  # dict
sonar_model = 'EK60'  # str
seabed_channel = '38'  # str
seabed_max_slope_deg = 30.0  # float
seabed_max_gap_s = 600.0  # float
seafloor_buffer_m = 10.0  # float
seabed_line = 'integration'  # str
surface_depth_m = 10.0  # float
frequencies_to_mask = [70, 120, 200]  # list
noise_range_sample_num = 10  # int
noise_ping_num = 5  # int
noise_snr_threshold = '3.0dB'  # str
noise_background_max = None  # TODO: set this value  # str
range_bin = '2m'  # str
ping_time_bin = '10s'  # str
range_var = 'depth'  # str
nan_threshold = 0.99  # float
viewer_chunk_ping_time = 2048  # int
pyramid_range_var = 'depth'  # str
pyramid_levels = 8  # int
pyramid_segment_columns = 16384  # int
pyramid_aggregation = 'linear_mean'  # str

## Included: processing_lvl_0_user.yaml

Steps: query_ncei, scan_raw_config, record_raw_configs, standardize_cal, build_cal_mapping

### Step: `query_ncei`
**Op:** `query_ncei_data`

Query an NCEI acoustic data catalog for file listings within a time window. Returns a list of result records (file metadata), not the files themselves.

**Parameters:**
- `file_time_start` str - ISO 8601 start of window, e.g. '2016-07-25T20:58'
- `file_time_end` str - ISO 8601 end of window
- `filters` dict - Key-value filters applied to NCEI metadata (e.g. CRUISE_NAME)
- `file_names` list - Keep only these files, matched on the D{date}-T{time} stamp in the name. A name the query did not return is an error. Empty keeps everything.


In [ ]:
# --- Parameters ---
_query_ncei__file_time_start = '2016-06-01T00:00:00'
_query_ncei__file_time_end = '2016-09-01T00:00:00'

_result = query_ncei_data(
    file_time_start=_query_ncei__file_time_start,
    file_time_end=_query_ncei__file_time_end,
    filters={'CRUISE_NAME': 'HB1603'},
)
ncei_results = _result['records']
query_label = _result['query_label']
raw_urls = _result['raw_urls']

### Step: `scan_raw_config`
**Op:** `read_raw_file_config`

Read one raw file's channel configuration from its datagram headers. Map it over the raw file paths and collect the results with record_raw_file_configs.

**Parameters:**
- `raw_file_path` path - One .raw file, as a local path or a remote URL. Normally wired to ${_item}.

- `verify_start_time` bool - If true, also read EK80 files in full to verify metadata_start_time, which is slower. The default single pass scan is enough for channel configuration.

- `verbose` bool - Print per-file progress. Set false on a long fan-out to keep the log readable.

- `max_scan_bytes` int - Most leading bytes of a remote EK60 or EK80 file to read instead of the whole file, in bytes. Unset reads whole files. Around 1 MiB suits EK60; FM EK80 needs tens of MiB. Too low only costs a whole-file read.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
file_config = []
for _item in raw_urls:
    _iter_file_config = read_raw_file_config(
        raw_file_path=_item,
        verbose=False,
        max_scan_bytes=1048576,
    )
    file_config.append(_iter_file_config)

### Step: `record_raw_configs`
**Op:** `record_raw_file_configs`

Collect per-file channel configurations into raw_file_configs/raw_file_configs.yaml under the calibration output directory. Use collect on the scan step's file_config output.

**Parameters:**
- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `verbose` bool - Print a summary of what was recorded.

In [ ]:
# --- Parameters ---
_record_raw_configs__output_base = './outputs/calibration'

_result = record_raw_file_configs(file_configs=file_config, output_base=_record_raw_configs__output_base)
raw_file_configs = _result['raw_file_configs']
frequencies = _result['frequencies']
raw_configs_path = _result['raw_configs_path']

### Step: `standardize_cal`
**Op:** `standardize_calibration_files`

Convert manufacturer calibration files (EK60 .cal or EK80 .xml) to standardized single-channel .yaml files under single_channel_calibration_files/. Skipped when the manufacturer files are unchanged.

**Parameters:**
- `cal_input_folder` path - Folder of manufacturer calibration files (.cal for EK60, .xml for EK80), local or a remote URL.

- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `cruise_id` str - Cruise identifier recorded in every generated single-channel file (e.g. "HB2407"), so a standardized file stays traceable to the survey it came from.

- `record_author` str - Name recorded as the author of every generated single-channel file. Not used in any calculation.

- `short_filenames` bool - Use compact names for the generated single-channel files. Set false for longer, more descriptive names.

- `overwrite` bool - Re-parse and rewrite even when the sidecar says the existing files are current, forcing a clean rebuild of the single-channel folder.

- `verbose` bool - Print the parsed parameters and the files written.
- `override_channels` dict - Write these channels as the single-channel files instead of parsing the manufacturer files. Same shape as the single_channel_data output: omit an entry to discard that channel, and edit values in place to correct them.


In [ ]:
# --- Parameters ---
_standardize_cal__cal_input_folder = '../calibration_files/HB201607_cal'
_standardize_cal__output_base = './outputs/calibration'
_standardize_cal__cruise_id = 'HB1603'
_standardize_cal__record_author = '<Your Name>'
_standardize_cal__short_filenames = True

_result = standardize_calibration_files(
    frequencies=frequencies,
    cal_input_folder=_standardize_cal__cal_input_folder,
    output_base=_standardize_cal__output_base,
    cruise_id=_standardize_cal__cruise_id,
    record_author=_standardize_cal__record_author,
    short_filenames=_standardize_cal__short_filenames,
)
single_channel_dir = _result['single_channel_dir']
channel_count = _result['channel_count']
skipped = _result['skipped']
standardize_cal_single_channel_data = _result['single_channel_data']

### Step: `build_cal_mapping`
**Op:** `build_calibration_mapping`

Match each raw channel to its calibration data and save the mapping files under mapping_files/. Mark the step regenerate: always in a recipe.

**Parameters:**
- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `single_channel_dir` path - Folder of standardized single-channel files. Defaults to single_channel_calibration_files/ under output_base. Wire it to the standardization step's single_channel_dir output.

- `raw_configs_path` path - Path of the saved raw_file_configs.yaml. Defaults to raw_file_configs/raw_file_configs.yaml under output_base, and is only consulted when raw_file_configs is not supplied.

- `conflict_resolution` str - What to do when one raw channel matches several calibration files. "error" (default) lists the conflicts and stops. "interactive" prompts on the terminal. "report" returns them on the conflicts output and writes no mapping.

- `calibration_choices` dict - Maps a conflict id from a previous run's conflicts output to a choice: the key to keep; a list of keys, or {"average": [keys]}, to average; or {"merge": [items]} to merge FM calibrations EK80 style, an item being a key or an average.

- `tolerances` dict - Per-field match tolerances, merged over the defaults (1 Hz for the frequencies, 1 W for transmit_power, 1e-6 s for transmit_duration_nominal). Widening one applies a calibration measured at a different setting.

- `unmapped_channels` str - The unmapped_channels policy, "warn" or "error", of the step that extracts calibration parameters from this mapping. Only labels the provenance report; unset says the outcome is decided downstream.

- `cruise_id` str - Recorded in the provenance report when given.
- `record_author` str - Recorded as the author of any record averaged or merged while resolving conflicts. When omitted, the combined calibrations' author is kept if they share one.

- `keep_unused` bool - Move calibration files no raw channel matched into unused_calibration_files/ instead of deleting them. They are the evidence when a channel comes out unmapped.

- `short_filenames` bool - Use compact calibration keys in the returned dictionaries and for averaged and merged records. Match the setting the single-channel files were written with.

- `verbose` bool - Print the mapping summary, preview, and verification results.

In [ ]:
# --- Parameters ---
_build_cal_mapping__output_base = './outputs/calibration'
_build_cal_mapping__single_channel_dir = single_channel_dir
_build_cal_mapping__raw_configs_path = raw_configs_path
_build_cal_mapping__short_filenames = True
_build_cal_mapping__keep_unused = True
_build_cal_mapping__conflict_resolution = 'error'

_result = build_calibration_mapping(
    raw_file_configs=raw_file_configs,
    output_base=_build_cal_mapping__output_base,
    single_channel_dir=_build_cal_mapping__single_channel_dir,
    raw_configs_path=_build_cal_mapping__raw_configs_path,
    short_filenames=_build_cal_mapping__short_filenames,
    tolerances={'transmit_power': 1001.0},
    keep_unused=_build_cal_mapping__keep_unused,
    conflict_resolution=_build_cal_mapping__conflict_resolution,
)
mapping_dict = _result['mapping_dict']
calibration_dict = _result['calibration_dict']
missing_params = _result['missing_params']
unused_files = _result['unused_file_names']
provenance = _result['provenance']
provenance_path = _result['provenance_path']
conflicts = _result['conflicts']
build_cal_mapping_single_channel_data = _result['single_channel_data']

*End of included section: processing_lvl_0_user.yaml*

## Included: processing_lvl_1_per_file.yaml

Steps: read_raw, combine_raw

### Step: `read_raw`
**Op:** `read_raw_files`

Open raw echosounder files with echopype and write each one to an intermediate store. Returns the store paths, or in-memory EchoData objects when intermediate_format is "none".

**Parameters:**
- `sonar_model` str - Sonar model string recognized by echopype (e.g. 'EK60', 'EK80')
- `include_bot` bool - Include bottom-detection (.bot) data if available
- `intermediate_format` str - Store format written between reading and combining. "netcdf" (default, recommended) writes temporary .nc files. "none" keeps everything in memory. "zarr" writes zarr stores, which can live on a gs:// temp dir.

- `use_swap` str - Whether echopype swaps data to disk while reading. "auto" (default) swaps a file whose expanded data would exceed about 40% of total RAM.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `combine_raw`
**Op:** `combine_raw_files`

Combine the per-file stores or EchoData objects from read_raw_files into one EchoData object. Mark the step checkpoint: always so the combined output is written once and reused.

**Parameters:**
- `ping_time_chunk` int - Chunk length along ping_time for the combined EchoData and its checkpoint, in pings. Larger values write faster to gs:// but use more memory; try 2000 to 4000 for bucket caches.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `extract_cal_params`
**Op:** `extract_standardized_cal_params`

Extract calibration parameters (gain, sa-correction, EBA) and environmental parameters (sound speed, absorption) from standardized calibration files, cross-referenced against the EchoData object for channel alignment.

**Parameters:**
- `raw_file_path` path - Raw file whose mapping to use, matched by basename. Wire to ${_item} in a per-file chain. Unset uses the first file in the survey, which suits whole-survey recipes.

- `unmapped_channels` str - What to do with a channel no calibration file matched. "warn" (default) reports it and uses the values the raw file records. "error" stops the run.

- `output_logs_folder` path - Where the fallback writes its calibration flags. Unset uses a temporary folder.

- `waveform_mode` str - The waveform_mode compute_sv uses, "CW" or "BB". Selects which beam group's channels are extracted, for EK80 files mixing CW and FM. Unset uses the first beam group.

- `encode_mode` str - The encode_mode the compute_sv step passes to echopype, "complex" or "power". Set it together with waveform_mode, or leave both unset.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_sv`
**Op:** `compute_sv`

Compute volume backscattering strength (Sv) from raw EchoData, applying cal_params and env_params when given and the raw file's own values otherwise.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_transducer_depth`
**Op:** `compute_transducer_depth`

Compute per-ping transducer depth in metres below the water surface from the EK Platform group, for feeding ``ep_add_depth`` as its ``depth_offset``. Use it instead of ``use_platform_vertical_offsets``, which gets the sign wrong.

**Parameters:**
- `datum_correction_m` float (m) - Metres added to the static offset when it is measured from a hull datum rather than the waterline. Positive pushes the transducer deeper. The default 0.0 takes the recorded offset as waterline referenced.

- `use_heave` bool - Add the per-ping ``vertical_offset`` (heave) term. False holds the static keel depth on every ping. Without usable heave the step says so and uses the static depth.

- `heave_sign` float - Sign applied to ``vertical_offset`` (1.0 or -1.0). The default was verified on HB2407 (EK80); check it on a new vessel or instrument setup.

- `interp_method` str - Interpolation used to put heave on ping_time, such as "linear" (default) or "nearest".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `ep_add_depth`
**Op:** `ep_add_depth`

Add a ``depth`` variable, in metres below the surface, to an Sv Dataset using echopype. Existing variables, including ``echo_range``, are kept.

**Parameters:**
- `depth_offset` (m) - Transducer depth in metres, as a constant or a per-ping DataArray such as ``compute_transducer_depth``'s output. When set, takes precedence over Platform vertical offsets.

- `tilt` float (degree) - Constant transducer tilt; 0 corresponds to a vertically-pointing transducer. When set, takes precedence over Platform/Beam angles.

- `downward` bool - True when transducers point downward (typical hull-mounted).
- `use_platform_vertical_offsets` bool - Use Echodata Platform group vertical offsets to compute transducer depth. EK60/EK80 only. Ignored when ``depth_offset`` is set.

- `use_platform_angles` bool - Use Echodata Platform group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_beam_angles``.

- `use_beam_angles` bool - Use Echodata Beam group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_platform_angles``.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `crop_survey_range`
**Op:** `crop_range`

Trim the deep range samples that hold no data after seafloor masking. Run it on the whole survey, after the per-file fan-in. Active criteria each give a cut and the shallowest wins.

**Parameters:**
- `max_range_m` float (m) - Hard cap on range. Samples deeper than this are dropped whether or not they hold data. Unset derives the crop from the data.

- `outlier_sigma` float - Standard deviations above the mean of each ping's deepest finite sample at which a ping counts as an outlier and is ignored. Unset disables outlier rejection and uses drop_trailing_all_nan.

- `outlier_margin` float - Multiplier on (mean + outlier_sigma x sd) giving the outlier threshold. Raise it to flag fewer pings. Only used when outlier_sigma is set.

- `drop_trailing_all_nan` bool - Crop to one past the deepest sample with a finite value anywhere (lossless). Ignored when outlier_sigma is set. False, with the other two unset, turns the step into a no-op.

- `data_var` str - Variable inspected for finite values.
- `range_var` str - Variable supplying each sample's range in metres.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_splitbeam_angle`
**Op:** `ep_add_splitbeam_angle`

Add split-beam ``angle_alongship`` and ``angle_athwartship`` variables to an Sv Dataset using echopype. Needed before the "blackwell" and phase seafloor detectors.

**Parameters:**
- `waveform_mode` str - Transmit waveform mode. Use ``CW`` for narrowband EK60/EK80 power/angle data, or ``BB`` for broadband complex samples.

- `encode_mode` str - Return echo encoding mode. Use ``power`` for EK60/EK80 power/angle data, or ``complex`` for complex samples.

- `pulse_compression` bool - Whether to pulse-compress broadband complex samples. Only valid for ``waveform_mode: BB`` and ``encode_mode: complex``.

- `to_disk` bool - Write split-beam angles back to disk when ``ds_Sv`` is a path. Keep false for in-memory recipe execution.

- `drop_last_hanning_zero` bool - Echopype compatibility option for broadband pulse compression. Usually leave false.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_crop`
**Op:** `crop_range`

Trim the deep range samples that hold no data after seafloor masking. Run it on the whole survey, after the per-file fan-in. Active criteria each give a cut and the shallowest wins.

**Parameters:**
- `max_range_m` float (m) - Hard cap on range. Samples deeper than this are dropped whether or not they hold data. Unset derives the crop from the data.

- `outlier_sigma` float - Standard deviations above the mean of each ping's deepest finite sample at which a ping counts as an outlier and is ignored. Unset disables outlier rejection and uses drop_trailing_all_nan.

- `outlier_margin` float - Multiplier on (mean + outlier_sigma x sd) giving the outlier threshold. Raise it to flag fewer pings. Only used when outlier_sigma is set.

- `drop_trailing_all_nan` bool - Crop to one past the deepest sample with a finite value anywhere (lossless). Ignored when outlier_sigma is set. False, with the other two unset, turns the step into a no-op.

- `data_var` str - Variable inspected for finite values.
- `range_var` str - Variable supplying each sample's range in metres.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_seabed_line`
**Op:** `phase_detect_seafloor`

Detect the seafloor from Sv and split-beam phase, returning the seabed echo's leading edge in metres per ping. Swaps in for the other seafloor ops, so ``create_seafloor_mask`` works unchanged; put any dead-zone offset in its ``seafloor_buffer_m``.

**Parameters:**
- `channel` str - Primary channel, as its label or its frequency in Hz or kHz as a string (e.g. "38"). Null picks the lowest frequency with enough seabed seeds.

- `r_min` float (m) - Lower bound of the search window on the dataset's range reference. Default 10 m past the first sample, which skips the transmit pulse.

- `r_max` float (m) - Upper bound of the search window. Default the end of the channel's range. Set both bounds on deep surveys to keep the run fast.

- `sigma_prior_m` float (m) - Uncertainty of z_prior; default 2 % of z_prior. Ignored without it.

- `regime` str - Depth regime preset: "shelf" (< 200 m), "slope" (200 to 1000 m), "deep" (> 1000 m) or "auto" from the search window midpoint. Sets the SNR reference, the seed Sv floor and the Mode 2 shape window.

- `mode` int - Scoring mode. 1 scores the six point features; 2 adds three local shape features, a little slower, which help where a school sits on the bottom.

- `line` str - Line to return: "leading_edge" (default), the start of the seabed echo, or "integration", the backstepped line with slope correction.

- `prior` str - "auto" (default) runs the Mode 0 slab estimator when no z_prior is given and uses it as the search window; "none" searches only the fixed r_min / r_max window.

- `missing` str - Value for pings with no seabed found after gap filling. "nan" (default) masks the whole ping downstream, for analysis. "max_range" uses the last recorded range, keeping the ping for survey pictures.

- `min_seabed_sv_db` float (dB) - Minimum mean Sv over one pulse length below a pick for it to count as seabed. Null disables the check.

- `prior_budget` int - Number of slabs (small groups of pings) the prior estimator reads.

- `n_candidates` int - Score maxima kept per ping for the line search.
- `max_slope_deg` float (deg) - Maximum plausible seabed slope; larger depth jumps between pings are penalized. Use 45 on rocky or canyon terrain.

- `alpha` float - Transition strength multiplier in [0.5, 2].
- `beta_fraction` float - Shallow-first tie-break prior as a fraction of the candidate score spread; 0 disables it. Targets ghost bottoms and the echo tail.

- `min_score` float - Candidates below this score are dropped; a ping with none is skipped and bridged. Scores are sums of tanh terms, so 0 means neutral.

- `walkback_db` float (dB) - Drop below the echo peak that marks the leading edge.
- `effective_beamwidth_deg` float (deg) - Effective beamwidth for the backstepped integration line in the diagnostics; default nominal times 12/7 (the ES38B ratio).

- `vessel_speed_m_s` float (m/s) - Vessel speed for along-track distance when ds_Sv has no latitude and longitude. Set it for transects without position data.

- `min_confidence` float - Pings whose posterior confidence is below this are flagged.
- `phase_weight` float - Weight of the two phase features. Raise to 1.5 where fish aggregations are the main confuser.

- `max_gap_s` float (s) - Interior runs of skipped pings up to this long are interpolated; longer runs stay NaN. Null interpolates nothing.

- `transition` str - "huber" or "truncated" (quadratic capped at the break-point, for cliffs).
- `pulse_length_s` float (s) - Pulse duration when ds_Sv has no tau_effective.
- `beamwidth_deg` float (deg) - Nominal beamwidth when ds_Sv has none.
- `detect_aliases` bool - Detect aliased seabed echoes on the primary channel and exclude them. Off by default; it can mask the true seabed in noisy water.

- `phase_threshold_deg2` float (deg2) - Seed phase-stability threshold on the summed angle variance within one pulse length; default (beamwidth / 2)^2.

- `confidence_tolerance_pulses` float - Alternatives within this many pulse lengths of the pick (plus the beam-geometry spread, or inside the same loud run) count as the same pick when computing confidence and margin.

- `diagnostics_path` str - Optional zarr path for the full diagnostics (features, scores, candidates, lines, confidence and flags). Local runs only.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_seabed_mask`
**Op:** `create_seafloor_mask`

Build a seafloor mask from baseline Sv and detected seafloor depth. The seafloor depth is passed as an explicit input, enabling swappable detection methods without modifying masking logic.

**Parameters:**
- `seafloor_buffer_m` float (m) - Height above the seafloor line to also mask, covering detection error and the near-bottom dead zone at the cost of some real backscatter.

- `range_var` str - Meter-valued variable on ds_Sv compared against the seafloor line, on the same vertical reference. Null (default) uses "depth" if present, otherwise "echo_range".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_surface_mask`
**Op:** `create_surface_mask`

Build a surface-interference mask from baseline Sv using a selectable method.

**Parameters:**
- `surface_depth_m` float (m) - Depth above which samples are masked out, removing the transmit pulse ringdown and near-surface bubble layer. The default of 0.0 masks nothing.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_frequency_mask`
**Op:** `create_frequency_mask`

Build a mask that excludes selected frequency channels. Implementation can be swapped to use different exclusion criteria.

**Parameters:**
- `frequencies_to_mask` list (kHz) - Nominal channel frequencies to exclude, in kHz (e.g. [70, 120, 200]). Every sample on a listed channel is marked invalid. The default empty list masks nothing.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_combine_masks`
**Op:** `combine_masks`

Combine two or more mask datasets into one final mask. Accepts a list of mask inputs to support inserting additional custom mask-producing steps into the chain without changing core framework code.

**Parameters:**
- `mode` str - Combination mode: 'and' (all must be valid) or 'or' (any must be valid)

*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_apply_mask`
**Op:** `apply_sv_mask`

Apply a boolean mask to an Sv Dataset, setting masked samples to NaN.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_remove_noise`
**Op:** `remove_background_noise`

Estimate and remove background noise from Sv using the De Robertis and Higginbottom method. Returns a cleaned Sv Dataset whose Sv holds the corrected values.

**Parameters:**
- `ping_num` int - Number of pings used to estimate the noise level
- `range_sample_num` int - Number of range samples used to estimate the noise level
- `background_noise_max` str - Upper limit on the estimated noise level (e.g. '-125dB'); omit to leave it unbounded
- `SNR_threshold` str - Signal-to-noise ratio below which corrected samples are masked out (e.g. '3.0dB')

*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_mask_sparse`
**Op:** `mask_sparse_bins`

Remove spatiotemporal bins with fewer valid samples than a threshold, reducing noise in MVBS computation.

**Parameters:**
- `range_bin` str - Range bin size (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size (e.g. '10s')
- `nan_threshold` float - Fraction of NaN values above which a bin is masked (0.0 to 1.0)
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth). Must match compute_mvbs and compute_per_cell_statistics.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `viewer_file_mvbs`
**Op:** `compute_mvbs`

Compute Mean Volume Backscattering Strength by averaging Sv over range and time bins. Returns a gridded MVBS Dataset.

**Parameters:**
- `range_bin` str (m) - Depth bin size for averaging (e.g. '2m')
- `ping_time_bin` str (s) - Ping-time bin size for averaging (e.g. '10s')
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth, which ep_add_depth adds).

- `range_var_max` str (m) - Deepest value the bins cover, as a unit string (e.g. '2100m'). Unset reads the maximum from the data; supplying it keeps the step lazy.

- `closed` str - Which side of each bin interval is closed: 'left' or 'right'
- `skipna` bool - Skip NaN values when averaging. When false, any NaN in a bin makes the whole bin NaN, so masked data collapses.

- `fill_value` float - Value for bins holding no data. Unset keeps echopype's default, NaN.

- `method` str - Flox reduction strategy for dask arrays, normally 'map-reduce'. Any other value needs reindex set to null in the same params block.

- `reindex` bool - Reindex at the blockwise stage rather than after it. False uses less memory and more time. Only meaningful when method is 'map-reduce'.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
raw_stores = []
echodata = []
cal_params = []
env_params = []
other_params = []
compute_sv_ds_Sv = []
transducer_depth = []
ep_add_depth_ds_Sv = []
crop_survey_range_ds_Sv = []
viewer_splitbeam_angle_ds_Sv = []
viewer_crop_ds_Sv = []
seafloor_depth = []
viewer_seabed_mask_mask = []
viewer_surface_mask_mask = []
viewer_frequency_mask_mask = []
viewer_combine_masks_mask = []
viewer_apply_mask_ds_Sv = []
viewer_remove_noise_ds_Sv = []
viewer_mask_sparse_ds_Sv = []
ds_MVBS = []
for _item in raw_urls:
    _iter_raw_stores = read_raw_files_to_stores(
        raw_file_paths=[_item],
        sonar_model='EK60',
        include_bot=False,
        intermediate_format='zarr',
    )
    raw_stores.append(_iter_raw_stores)
    _iter_echodata = combine_raw_stores(raw_stores=_iter_raw_stores)
    echodata.append(_iter_echodata)
    _result = extract_standardized_calibration_parameters(
        calibration_dict=calibration_dict,
        mapping_dict=mapping_dict,
        echodata=_iter_echodata,
        raw_file_path=_item,
    )
    _iter_cal_params = _result['cal_params']
    _iter_env_params = _result['env_params']
    _iter_other_params = _result['other_params']
    cal_params.append(_iter_cal_params)
    env_params.append(_iter_env_params)
    other_params.append(_iter_other_params)
    _iter_compute_sv_ds_Sv = compute_Sv(
        echodata=_iter_echodata,
        cal_params=_iter_cal_params,
        env_params=_iter_env_params,
    )
    compute_sv_ds_Sv.append(_iter_compute_sv_ds_Sv)
    _iter_transducer_depth = compute_transducer_depth(
        ds_Sv=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        use_heave=True,
        heave_sign=1.0,
    )
    transducer_depth.append(_iter_transducer_depth)
    _iter_ep_add_depth_ds_Sv = add_depth(
        ds=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        depth_offset=_iter_transducer_depth,
    )
    ep_add_depth_ds_Sv.append(_iter_ep_add_depth_ds_Sv)
    _iter_crop_survey_range_ds_Sv = crop_range_samples(
        ds_Sv=_iter_ep_add_depth_ds_Sv,
        max_range_m=1800.0,
        drop_trailing_all_nan=False,
        outlier_sigma=None,
    )
    crop_survey_range_ds_Sv.append(_iter_crop_survey_range_ds_Sv)
    _iter_viewer_splitbeam_angle_ds_Sv = add_splitbeam_angle(
        source_Sv=_iter_ep_add_depth_ds_Sv,
        echodata=_iter_echodata,
        waveform_mode='CW',
        encode_mode='power',
        to_disk=False,
    )
    viewer_splitbeam_angle_ds_Sv.append(_iter_viewer_splitbeam_angle_ds_Sv)
    _iter_viewer_crop_ds_Sv = crop_range_samples(
        ds_Sv=_iter_viewer_splitbeam_angle_ds_Sv,
        max_range_m=3000.0,
        drop_trailing_all_nan=False,
        outlier_sigma=None,
    )
    viewer_crop_ds_Sv.append(_iter_viewer_crop_ds_Sv)
    _iter_seafloor_depth = detect_seafloor_phase(
        ds_Sv=_iter_viewer_crop_ds_Sv,
        channel='38',
        max_slope_deg=30.0,
        max_gap_s=600.0,
        line='integration',
        missing='max_range',
    )
    seafloor_depth.append(_iter_seafloor_depth)
    _iter_viewer_seabed_mask_mask = create_seafloor_mask(
        ds_Sv=_iter_viewer_crop_ds_Sv,
        seafloor_depth=_iter_seafloor_depth,
        seafloor_buffer_m=10.0,
    )
    viewer_seabed_mask_mask.append(_iter_viewer_seabed_mask_mask)
    _iter_viewer_surface_mask_mask = create_surface_mask(ds_Sv=_iter_viewer_crop_ds_Sv, surface_depth_m=10.0)
    viewer_surface_mask_mask.append(_iter_viewer_surface_mask_mask)
    _iter_viewer_frequency_mask_mask = create_frequency_mask(ds_Sv=_iter_viewer_crop_ds_Sv, frequencies_to_mask=[70, 120, 200])
    viewer_frequency_mask_mask.append(_iter_viewer_frequency_mask_mask)
    _iter_viewer_combine_masks_mask = combine_masks(masks=[_iter_viewer_surface_mask_mask, _iter_viewer_frequency_mask_mask, _iter_viewer_seabed_mask_mask], mode='and')
    viewer_combine_masks_mask.append(_iter_viewer_combine_masks_mask)
    _iter_viewer_apply_mask_ds_Sv = apply_mask_to_sv(ds_Sv=_iter_viewer_crop_ds_Sv, mask=_iter_viewer_combine_masks_mask)
    viewer_apply_mask_ds_Sv.append(_iter_viewer_apply_mask_ds_Sv)
    _iter_viewer_remove_noise_ds_Sv = remove_noise(
        ds_Sv=_iter_viewer_apply_mask_ds_Sv,
        range_sample_num=10,
        ping_num=5,
        SNR_threshold='3.0dB',
        background_noise_max=noise_background_max,
    )
    viewer_remove_noise_ds_Sv.append(_iter_viewer_remove_noise_ds_Sv)
    _iter_viewer_mask_sparse_ds_Sv = mask_sparse_bins(
        ds_Sv=_iter_viewer_remove_noise_ds_Sv,
        range_bin='2m',
        ping_time_bin='10s',
        range_var='depth',
        nan_threshold=0.99,
    )
    viewer_mask_sparse_ds_Sv.append(_iter_viewer_mask_sparse_ds_Sv)
    _iter_ds_MVBS = compute_MVBS(
        ds_Sv=_iter_viewer_mask_sparse_ds_Sv,
        range_bin='2m',
        ping_time_bin='10s',
        range_var='depth',
        range_var_max='3010m',
    )
    ds_MVBS.append(_iter_ds_MVBS)

*End of included section: processing_lvl_1_per_file.yaml*

## Included: processing_lvl_2_survey.yaml

Steps: extract_cal_params, compute_sv, compute_transducer_depth, ep_add_depth, crop_survey_range

*End of included section: processing_lvl_2_survey.yaml*

## Included: processing_lvl_2_viewer_seabed.yaml

Steps: viewer_splitbeam_angle, viewer_crop, viewer_seabed_line, viewer_seabed_mask, viewer_surface_mask, viewer_frequency_mask, viewer_combine_masks, viewer_apply_mask, viewer_remove_noise

*End of included section: processing_lvl_2_viewer_seabed.yaml*

## Included: processing_lvl_3_viewer.yaml

Steps: viewer_mask_sparse, viewer_file_mvbs, merge_viewer_mvbs, rechunk_viewer_mvbs

### Step: `merge_viewer_mvbs`
**Op:** `merge_datasets`

Concatenate a collected list of Datasets into one along ``dim``, for example per-file MVBS gathered with ``collect``. A single Dataset passes through unchanged.

**Parameters:**
- `dim` str - Dimension to concatenate the collected Datasets along.
- `on_duplicate` str - How to handle two collected segments with the same label on ``dim``. Unset keeps both (fine when segments can't overlap). "mean" averages them (binned products split at a segment boundary). "first" keeps the first and sorts (overlapping slices of one store).


In [ ]:
# --- Parameters ---
_merge_viewer_mvbs__dim = 'ping_time'
_merge_viewer_mvbs__on_duplicate = 'mean'

merged = concat_datasets(
    datasets=ds_MVBS,
    dim=_merge_viewer_mvbs__dim,
    on_duplicate=_merge_viewer_mvbs__on_duplicate,
)

### Step: `rechunk_viewer_mvbs`
**Op:** `rechunk_dataset`

Rechunk a Dataset along ping_time so the steps after it can run dask-parallel.

**Parameters:**
- `ping_time_chunk` int - Target chunk size along ping_time.

In [ ]:
# --- Parameters ---
_rechunk_viewer_mvbs__ping_time_chunk = 2048

rechunk_viewer_mvbs_ds_Sv = rechunk_dataset(ds_Sv=merged, ping_time_chunk=_rechunk_viewer_mvbs__ping_time_chunk)

*End of included section: processing_lvl_3_viewer.yaml*

## Included: viewer_pyramid.yaml

Steps: plan_segments, build_pyramid, survey_echogram_store

### Step: `plan_segments`
**Op:** `plan_pyramid_segments`

Choose ping ranges that a pyramid can be built over independently and then concatenated exactly. Every range but the last is a multiple of 2**(levels-1) pings.

**Parameters:**
- `levels` int - Levels the build will produce, which is what sets the grid.
- `target_pings` int - Rough segment size in pings, rounded down onto the grid. Raise it for a machine with more memory, lower it for a deep water column.


In [ ]:
# --- Parameters ---
_plan_segments__levels = 8
_plan_segments__target_pings = 16384

_result = plan_pyramid_segments(
    ds_Sv=rechunk_viewer_mvbs_ds_Sv,
    levels=_plan_segments__levels,
    target_pings=_plan_segments__target_pings,
)
ranges = _result['ranges']
grid = _result['grid']

### Step: `build_pyramid`
**Op:** `build_echogram_pyramid`

Build a level-of-detail pyramid from an Sv Dataset for the WebGPU echogram viewer, where level k merges 2**k pings. `aa-echogram serve --store <path>` reads a checkpointed pyramid straight from the cache.

**Parameters:**
- `levels` int - Levels to build, halving the ping axis each time. Eight levels reduce the ping axis by 128 at the coarsest.

- `range_var` str - Vertical coordinate to build the geometry from. Use echo_range, not depth; the surface reference is kept as a per ping transducer depth.

- `ping_range` list - Optional [start, stop] naming one segment of the ping axis. Supplied by a mapped step as ${_item} from plan_pyramid_segments.

- `aggregation` str - How pings are merged: 'linear_mean' averages in linear space and reports decibels (right for biomass); 'max' keeps the loudest sample, preserving a single strong target.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
build_pyramid_pyramid = []
for _item in ranges:
    _iter_build_pyramid_pyramid = build_echogram_pyramid(
        ds_Sv=rechunk_viewer_mvbs_ds_Sv,
        ping_range=_item,
        levels=8,
        range_var='depth',
        aggregation='linear_mean',
    )
    build_pyramid_pyramid.append(_iter_build_pyramid_pyramid)

### Step: `survey_echogram_store`
**Op:** `merge_echogram_pyramids`

Collect per segment pyramids into one, level by level. Only correct when the parts were cut by plan_pyramid_segments.

**Parameters:**
- `dim` str - Ping dimension to concatenate along.

In [ ]:
# --- Parameters ---
_survey_echogram_store__dim = 'ping_time'

survey_echogram_store_pyramid = merge_echogram_pyramids(parts=build_pyramid_pyramid, dim=_survey_echogram_store__dim)

*End of included section: viewer_pyramid.yaml*